In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.preprocessing import StandardScaler
import joblib
import json
from sklearn.multioutput import MultiOutputRegressor
from sklearn.svm import SVR
import shutup; shutup.please()

# Functions

In [2]:
def naive_forecaster(X_test):
    lag1_cols = [f'DA_price lag1_hour{i}' for i in range(24)]
    lag7_cols = [f'DA_price lag7_hour{i}' for i in range(24)]
    prediction = []
    for index, row in X_test.iterrows():
        if pd.to_datetime(index).weekday() in [1,2,3,4,6]:
            prediction.append(row[lag1_cols].values)
        else:
            prediction.append(row[lag7_cols].values)
    prediction = np.array(prediction)
    return prediction

# calculate smape
def smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

#day average error
def dae(y_true, y_pred):
    return np.mean(np.abs(np.mean(y_pred,axis=1) - np.mean(y_true,axis=1)))

def rmae(y_true, y_pred, y_pred_naive):
    mae_pred = np.mean(np.abs(y_pred - y_true))
    mae_naive = np.mean(np.abs(y_pred_naive - y_true))
    return mae_pred/mae_naive

In [3]:
period=3
print(f'{2015+period}-01-08',f'{2020+period}-12-31')

2018-01-08 2023-12-31


In [4]:
def get_best_params_and_test(country, period=0, n_trials=50):

    # Load the data
    inputs = pd.read_csv(os.path.join("cut_data", country, "inputs.csv"),index_col=0)
    inputs = inputs.fillna(value=0)
    outputs = pd.read_csv(os.path.join("cut_data", country, "outputs.csv"),index_col=0)

    X_train = inputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']
    y_train = outputs.loc[f'{2015+period}-01-09':f'{2018+period}-12-31']

    X_val = inputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']
    y_val = outputs.loc[f'{2019+period}-01-01':f'{2019+period}-12-31']

    X_test = inputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']
    y_test = outputs.loc[f'{2020+period}-01-01':f'{2020+period}-12-31']

    # Scaling
    scaler1, scaler2 = StandardScaler(), StandardScaler()

    X_train = pd.DataFrame(scaler1.fit_transform(X_train), columns=X_train.columns)
    X_val = pd.DataFrame(scaler1.transform(X_val), columns=X_val.columns)
    X_test = pd.DataFrame(scaler1.transform(X_test), columns=X_test.columns)

    y_train = pd.DataFrame(scaler2.fit_transform(y_train), columns=y_train.columns)
    y_val = pd.DataFrame(scaler2.transform(y_val), columns=y_val.columns)
    y_test = pd.DataFrame(scaler2.transform(y_test), columns=y_test.columns)


    def objective(trial):
        # Define the hyperparameters to tune for SVM
        C = trial.suggest_float('C', 1e-6, 1e+6, log=True)
        epsilon = trial.suggest_float('epsilon', 1e-6, 1e-1, log=True)
        kernel = trial.suggest_categorical('kernel', ['rbf', 'poly', 'sigmoid'])
        degree = trial.suggest_int('degree', 2, 5) if kernel == 'poly' else 3  # Default to 3 for non-poly kernels
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto']) if kernel != 'linear' else 'scale'

        # Define the SVR model with the suggested hyperparameters
        model = MultiOutputRegressor(SVR(
            C=C,
            epsilon=epsilon,
            kernel=kernel,
            degree=degree,
            gamma=gamma,
            cache_size=2000,  # Increased cache size for potentially better performance
            max_iter=10000
        ))

        # Fit the model
        model.fit(X_train, y_train)

        # Predict on validation set
        y_pred = model.predict(X_val)

        # Flatten the predicted and actual values
        y_pred_flattened = y_pred.flatten()
        y_val_flattened = y_val.values.flatten()
        y_pred_naive_flattened = naive_forecaster(X_val).flatten()

        # Calculate mean absolute error between flattened vectors
        rmae_val = rmae(y_val_flattened, y_pred_flattened, y_pred_naive_flattened)

        # Return the MAE as the objective to minimize
        return rmae_val

    # Create an Optuna study
    study = optuna.create_study(direction='minimize')

    # Optimize the hyperparameters
    study.optimize(objective, n_trials=n_trials, n_jobs=-1)  # You can adjust the number of trials

    # Since SVR doesn't directly use these parameters, we'll manually construct the model
    best_params = study.best_params
    best_model = MultiOutputRegressor(SVR(
        C=best_params['C'],
        epsilon=best_params['epsilon'],
        kernel=best_params['kernel'],
        degree=best_params['degree'] if best_params['kernel'] == 'poly' else 3,  # Default to 3 if not poly
        gamma=best_params['gamma'],
        cache_size=2000,
        max_iter=10000
    ))

    X_train_val = pd.concat([X_train, X_val])
    y_train_val = pd.concat([y_train, y_val])

    best_model.fit(X_train_val, y_train_val)
    
    y_pred = best_model.predict(X_test)

    y_pred_naive = naive_forecaster(X_test)
    y_test_val = y_test.values

    # inverse transform y_pred, y_pred_naive, y_test_val
    y_pred = scaler2.inverse_transform(y_pred)
    y_pred_naive = scaler2.inverse_transform(y_pred_naive)
    y_test_val = scaler2.inverse_transform(y_test_val)
    
    smape_score = smape(y_test_val.flatten(), y_pred.flatten())
    mae_score = mean_absolute_error(y_test_val.flatten(), y_pred.flatten())
    dae_score = dae(y_test_val, y_pred)
    rmae_score = rmae(y_test_val, y_pred, y_pred_naive)
    
    naiv_smape = smape(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_mae = mean_absolute_error(y_test_val.flatten(), y_pred_naive.flatten())
    naiv_dae =  dae(y_test_val, y_pred_naive)
    
    
    return [country, period, smape_score, mae_score, dae_score, rmae_score, naiv_smape, naiv_mae, naiv_dae, str(study.best_params)]

# Training and evaluation

In [ ]:
country_name_list = sorted(os.listdir(os.path.join('cut_data')))

final_results = {}

for country in tqdm(country_name_list):
    for period in [0,4]:
        results = get_best_params_and_test(country, period, n_trials=50)
        final_results[f"{country}_{period}"] = results
        # save as json
        with open('svr_final_results.json', 'w') as f:
            json.dump(final_results, f)
    #print(f'{country} is done')

100%|██████████| 22/22 [14:22:48<00:00, 2353.13s/it]  


In [6]:
import json
import numpy as np
import pandas as pd

with open('svr_final_results.json', 'r') as f:
    final_results = json.load(f)

df = pd.DataFrame(final_results).T
df.columns = ['country', 'period', 'smape', 'mae', 'dae', 'rmae', 'naive_smape', 'naive_mae', 'naive_dae', 'best_params']
df.index = np.arange(len(df))
df.to_csv('svr_final_results.csv', index=False)